# Task 9-1


## Setup


In [1]:
!pip install transformers accelerate -q


In [2]:
from transformers import pipeline, GenerationConfig
import torch

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print("Loading model...")
pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
)
tokenizer = pipe.tokenizer

def gen_cfg(max_new_tokens=450):
    return GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )

def run_prompt(text, max_new_tokens=450):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Be clear and direct."},
        {"role": "user", "content": text},
    ]
    out = pipe(messages, generation_config=gen_cfg(max_new_tokens))
    return out[0]["generated_text"][-1]["content"].strip()

def run_all(prompts):
    results = {}
    n = len(prompts)
    for i, (name, p) in enumerate(prompts.items(), 1):
        print(f"[{i}/{n}] {name}", flush=True)
        results[name] = run_prompt(p)
    return results

def show(title, text):
    print()
    print("---", title, "---")
    print(text)
    print()

print("Device:", pipe.device)


Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Device: cuda:0


### Prompt 1 — supply chains


In [3]:
prompts = {
    "ZERO-SHOT": "Explain why supply chains fail",

    "FEW-SHOT": """
Here are examples of system failure explanations:
Q: Why do bridges fail? A: Bridges fail due to material fatigue, design flaws, overloading, and lack of maintenance.
Q: Why do businesses fail? A: Businesses fail due to poor cash flow, lack of market demand, bad management, and competition.
Now answer in the same style:
Q: Why do supply chains fail?
""",

    "CHAIN OF THOUGHT": """
Explain why supply chains fail.
Think step by step, starting from the supplier, moving through manufacturing,
logistics, distribution, and ending at the customer.
Identify failure points at each stage.
""",

    "ROLE PROMPTING": """
You are a senior supply chain consultant with 20 years of experience
advising Fortune 500 companies including Amazon and Toyota.
Explain why supply chains fail, drawing on real-world examples.
""",

    "SELF-CONSISTENCY A": "What causes supply chain failures?",
    "SELF-CONSISTENCY B": "Why do supply chains break down?",
    "SELF-CONSISTENCY C": "What are the root causes of supply chain disruptions?",

    "TREE OF THOUGHT": """
Explain why supply chains fail by exploring three different perspectives:
Path 1: From the supplier point of view
Path 2: From the logistics and transportation point of view
Path 3: From the demand and customer point of view
After exploring all three paths, synthesise them into one unified conclusion.
""",

    "CONSTRAINED (JSON)": """
Explain why supply chains fail.
Return your answer strictly as valid JSON with exactly these keys:
{\"causes\": [], \"real_world_examples\": [], \"prevention_strategies\": []}
Each key should contain a list of 3 items. Return only the JSON, no other text.
"""
}

out = run_all(prompts)
for k in prompts:
    show(k, out[k])


[1/9] ZERO-SHOT
[2/9] FEW-SHOT
[3/9] CHAIN OF THOUGHT
[4/9] ROLE PROMPTING
[5/9] SELF-CONSISTENCY A
[6/9] SELF-CONSISTENCY B
[7/9] SELF-CONSISTENCY C
[8/9] TREE OF THOUGHT
[9/9] CONSTRAINED (JSON)

--- ZERO-SHOT ---
Supply chains fail when there are various problems in the process of acquiring, transporting, and delivering the goods to the end-user. Here are some reasons why supply chains fail:

1. Lack of coordination: When different parties in the supply chain are not aware of each other's activities, there can be confusion and disruption in the process. This can lead to delays in delivery, quality issues, and reduced efficiency.

2. Lack of visibility: The lack of visibility in the supply chain can lead to delays, incorrect products, and incorrect orders. This can result in lost sales opportunities, reduced profits, and a decrease in customer satisfaction.

3. Insufficient quality control: Supply chain management can result in a loss of quality control over the product throughout it

### Prompt 2 — sheep riddle


In [4]:
prompts = {
    "ZERO-SHOT": "A farmer has 17 sheep. All but 9 run away. How many sheep are left?",

    "FEW-SHOT": """
Here are examples of word problem solutions:
Q: A baker has 10 cakes. All but 4 are sold. How many remain? A: 4 remain. "All but 4" means 4 are kept.
Q: A library has 50 books. All but 20 are borrowed. How many are left? A: 20 remain. "All but 20" means 20 stay.

Now solve:
Q: A farmer has 17 sheep. All but 9 run away. How many sheep are left?
""",

    "CHAIN OF THOUGHT": """
A farmer has 17 sheep. All but 9 run away. How many sheep are left?
Think through this carefully step by step before giving your answer.
""",

    "ROLE PROMPTING": """
You are a mathematics teacher who is very careful about the precise meaning of words in problems.
Solve this problem: A farmer has 17 sheep. All but 9 run away. How many sheep are left?
Explain your reasoning clearly.
""",

    "SELF-CONSISTENCY A": "If a farmer starts with 17 sheep and all but 9 escape, what is the remaining count?",
    "SELF-CONSISTENCY B": "17 sheep are in a field. All except 9 run away. How many sheep remain in the field?",
    "SELF-CONSISTENCY C": "A farmer owns 17 sheep. All but 9 of them run away. What number of sheep does he still have?",

    "TREE OF THOUGHT": """
Solve this problem by exploring two interpretations:
Path A: Interpret 'all but 9' as meaning 9 sheep stay behind
Path B: Interpret 'all but 9' as meaning all sheep except 9 run away, so 17-9=8 leave
After exploring both, determine which interpretation is linguistically correct and give the final answer.
Problem: A farmer has 17 sheep. All but 9 run away. How many are left?
""",

    "CONSTRAINED (JSON)": """
A farmer has 17 sheep. All but 9 run away. How many sheep are left?
Return your answer strictly as JSON: {\"interpretation\": \"\", \"calculation\": \"\", \"answer\": 0}
Return only JSON, no other text.
"""
}

out = run_all(prompts)
for k in prompts:
    show(k, out[k])


[1/9] ZERO-SHOT
[2/9] FEW-SHOT


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[3/9] CHAIN OF THOUGHT
[4/9] ROLE PROMPTING
[5/9] SELF-CONSISTENCY A
[6/9] SELF-CONSISTENCY B
[7/9] SELF-CONSISTENCY C
[8/9] TREE OF THOUGHT
[9/9] CONSTRAINED (JSON)

--- ZERO-SHOT ---
The given text states, "All but 9 ran away." Therefore, based on the given scenario, 16 sheep are left, i.e., all except for the 9 sheep that ran away.


--- FEW-SHOT ---
"All but 9" means 18 sheep stay. The answer is 18.


--- CHAIN OF THOUGHT ---
To answer the question, we need to first determine the number of sheep the farmer has: 17. We know that all but 9 sheep ran away, so the number of sheep remaining is 16.

So, to answer the question, 16 sheep remain.


--- ROLE PROMPTING ---
Firstly, let's define what we mean by "run away": 

A sheep is said to "run away" if it escapes from its owner's fenced-in pasture or enclosure. Since the farmer has 17 sheep, not all of them are "run away", but only 9 of them have escaped.

Now, let's analyze the problem statement:

"A farmer has 17 sheep. All but 9 run aw

### Prompt 3 — two-sentence horror


In [5]:
prompts = {
    "ZERO-SHOT": "Write a two-sentence horror story",

    "FEW-SHOT": """
Here are examples of effective two-sentence horror stories:
Example 1: "The last thing I saw was my alarm clock flashing 12:07 before she pushed her long, rotting fingernails through my chest. It took me forever to realize that I had died when she was born."
Example 2: "I always thought my cat was staring at nothing. Then I got the glasses."

Now write a new original two-sentence horror story in the same style:
""",

    "CHAIN OF THOUGHT": """
Write a two-sentence horror story.
First, think step by step: decide on the setting, the character, the threat,
and the twist. Then write the story.
""",

    "ROLE PROMPTING": """
You are Stephen King, the master of psychological horror.
Your stories create dread through implication rather than gore.
Write a two-sentence horror story that leaves the reader deeply unsettled.
""",

    "SELF-CONSISTENCY A": "Write a very short horror story in exactly two sentences.",
    "SELF-CONSISTENCY B": "Create a two-line scary story that is genuinely chilling.",
    "SELF-CONSISTENCY C": "Give me a horrifying micro-story in two sentences.",

    "TREE OF THOUGHT": """
Write a two-sentence horror story. First explore three different horror approaches:
Approach 1: Supernatural horror (ghosts, monsters)
Approach 2: Psychological horror (reality questioned)
Approach 3: Domestic horror (danger in familiar places)
Then choose the most effective approach and write the final story.
""",

    "CONSTRAINED": """
Write a two-sentence horror story.
Requirements:
- Sentence 1: Set the scene (maximum 15 words)
- Sentence 2: The terrifying reveal (maximum 15 words)
- Must not mention blood, death, or monsters directly
- Must end on an implied threat
"""
}

out = run_all(prompts)
for k in prompts:
    show(k, out[k])


[1/9] ZERO-SHOT
[2/9] FEW-SHOT
[3/9] CHAIN OF THOUGHT
[4/9] ROLE PROMPTING
[5/9] SELF-CONSISTENCY A
[6/9] SELF-CONSISTENCY B
[7/9] SELF-CONSISTENCY C
[8/9] TREE OF THOUGHT
[9/9] CONSTRAINED

--- ZERO-SHOT ---
"As you approach the haunted house, the air becomes heavy with an eerie, otherworldly energy. You can feel the presence of something sinister lurking just beyond the walls. Suddenly, you hear a faint whispering that sounds like someone or something is trying to speak to you. Terrified, you back away, unsure if you're imagining things or if there's something truly sinister lurking in the shadows."


--- FEW-SHOT ---
Sarah had always been fascinated by the mysterious, glowing orb that hung from her cat's neck. But one night, while tucked up in bed, she suddenly heard a sickening pop, followed by an explosion.

Sarah's eyes widened as she stumbled out of bed, her heart racing. She could hear the cats meowing and snarling in the hallway, but she didn't care. She ran down the stairs, h

### Prompt 4 — diabetes symptoms (not medical advice)


In [6]:
prompts = {
    "ZERO-SHOT": "What are the symptoms of diabetes?",

    "FEW-SHOT": """
Here are examples of well-structured medical symptom descriptions:
Q: What are the symptoms of hypertension?
A: Hypertension often has no symptoms (silent killer), but may include: severe headache, fatigue, vision problems, chest pain, difficulty breathing, irregular heartbeat.

Q: What are the symptoms of anaemia?
A: Anaemia symptoms include: extreme fatigue, weakness, pale skin, shortness of breath, dizziness, cold hands and feet, brittle nails.

Now answer in the same structured format:
Q: What are the symptoms of diabetes?
""",

    "CHAIN OF THOUGHT": """
What are the symptoms of diabetes?
Think step by step: first consider Type 1 diabetes symptoms, then Type 2 diabetes symptoms,
then symptoms common to both, then explain why each symptom occurs physiologically.
""",

    "ROLE PROMPTING": """
You are an experienced endocrinologist explaining diabetes symptoms to a patient
who has just been referred to your clinic. Use clear, accessible language but
be medically precise. What are the symptoms of diabetes?
""",

    "SELF-CONSISTENCY A": "What signs and symptoms indicate a person may have diabetes?",
    "SELF-CONSISTENCY B": "How does diabetes present clinically? What does a diabetic patient experience?",
    "SELF-CONSISTENCY C": "List the warning signs of diabetes that someone should watch out for.",

    "TREE OF THOUGHT": """
Describe the symptoms of diabetes from three clinical perspectives:
Perspective 1: Early-stage symptoms that are easy to miss
Perspective 2: Advanced symptoms that indicate poor control
Perspective 3: Differences between Type 1 and Type 2 presentations
Synthesise all three into a complete clinical picture.
""",

    "CONSTRAINED (JSON)": """
What are the symptoms of diabetes?
Return your answer strictly as JSON:
{\"type_1_symptoms\": [], \"type_2_symptoms\": [], \"common_symptoms\": [], \"emergency_symptoms\": []}
Each list should contain 4 items. Return only the JSON, no other text.
"""
}

out = run_all(prompts)
for k in prompts:
    show(k, out[k])


[1/9] ZERO-SHOT
[2/9] FEW-SHOT
[3/9] CHAIN OF THOUGHT
[4/9] ROLE PROMPTING
[5/9] SELF-CONSISTENCY A
[6/9] SELF-CONSISTENCY B
[7/9] SELF-CONSISTENCY C
[8/9] TREE OF THOUGHT
[9/9] CONSTRAINED (JSON)

--- ZERO-SHOT ---
Some of the common symptoms of diabetes include:

1. Blurred vision
2. Dark urine or stool
3. Sore, tender, or swollen feet or legs
4. Frequent urination or increased thirst
5. Low blood sugar (hypoglycemia)
6. Numbness, tingling, or pain in the hands, feet, or legs
7. Red, itchy, or blistering skin
8. Fatigue
9. Dehydration (less than 60% of body weight)
10. Thinning of the blood (hypothyroidism)
11. Weight gain
12. Skin changes (such as brown-black skin discoloration)
13. Abnormal heart rhythms (such as atrial fibrillation)
14. Kidney damage
15. Tight, clammy skin (sweating) or a rash

Other symptoms may include fatigue, frequent headaches, or changes in vision. Diagnosing diabetes can be challenging, so it's always a good idea to consult with a medical professional to ge